## Cellule 0 — Configuration & patches

In [2]:
import torch, os, pickle, math, time, gzip
import numpy as np
import pandas as pd
import scipy.sparse
from collections import defaultdict
from tqdm import tqdm

# patches
_orig = torch.load
torch.load = lambda *a, **k: _orig(*a, **{**k, 'weights_only': False})
if not hasattr(scipy.sparse.dok_matrix, '_update'):
    def _u(self, d):
        for key,val in d.items(): dict.__setitem__(self, key, val)
    scipy.sparse.dok_matrix._update = _u

RECBOLE_ROOT = r'D:\Memoire\LLMKGER\recbole_data_large'
KG_DIR       = os.path.join(RECBOLE_ROOT, 'llmkger')
RESULTS_REC  = r'D:\Memoire\LLMKGER\reco_results_final'
LGCN_PATH    = r'D:\Memoire\LLMKGER\notebooks\saved\LightGCN-Jun-27-2026_18-31-14.pth'
KGE_DIR      = r'D:\Memoire\LLMKGER\kge_results_v2\RotatE_v2'
DEVICE       = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(42); np.random.seed(42)
print("Device:", DEVICE)

Device: cuda


## Étape 1 — Dataset RecBole + split exact (train/valid/test)

On recharge le dataset et on extrait le split **exact** de RecBole (le même que celui validé en Phase 5, où LightGCN reproduit 0.0833).

In [3]:
from recbole.config import Config
from recbole.data import create_dataset, data_preparation
from recbole.utils import init_seed, init_logger, get_model

base_config = {
    'data_path': RECBOLE_ROOT, 'dataset': 'llmkger',
    'USER_ID_FIELD':'user_id','ITEM_ID_FIELD':'item_id','RATING_FIELD':'rating',
    'load_col':{'inter':['user_id','item_id','rating']},
    'eval_args':{'split':{'LS':'valid_and_test'},'order':'RO','group_by':'user','mode':'full'},
    'metrics':['Recall','NDCG','MRR'],'topk':[10],'valid_metric':'NDCG@10',
    'epochs':100,'train_batch_size':2048,'eval_batch_size':2048,
    'learning_rate':0.001,'embedding_size':64,'n_layers':3,
    'stopping_step':10,'device':DEVICE,'seed':42,'show_progress':False,
}
config = Config(model='LightGCN', dataset='llmkger', config_dict=base_config)
init_seed(config['seed'], config['reproducibility']); init_logger(config)
dataset = create_dataset(config)
train_data, valid_data, test_data = data_preparation(config, dataset)
n_users = dataset.num(config['USER_ID_FIELD'])
n_items = dataset.num(config['ITEM_ID_FIELD'])
print(f"Users:{n_users:,} Items:{n_items:,}")

user2id = dict(dataset.field2token_id[config['USER_ID_FIELD']])
item2id = dict(dataset.field2token_id[config['ITEM_ID_FIELD']])
id2item = {v:k for k,v in item2id.items()}
id2user = {v:k for k,v in user2id.items()}
candidate_items = [i for i in range(n_items) if id2item.get(i,'[PAD]')!='[PAD]']
candidate_courses = [id2item[i] for i in candidate_items]
print(f"Cours candidats: {len(candidate_items):,}")

c:\Users\AMIRA\anaconda3\envs\llmkger\lib\site-packages\recbole\data\dataset\dataset.py:648: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  feat[field].fillna(value=0, inplace=True)
c:\Users\AMIRA\anaconda3\envs\llmkger\lib\site-packages\recbole\data\dataset\dataset.py:650: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are sett

Users:53,898 Items:2,813
Cours candidats: 2,812


In [4]:
# Split exact RecBole : cours test/valid + historique à masquer
def extract_split(dl):
    pos_item, mask = {}, {}
    for batch in dl:
        interaction, history_index, positive_u, positive_i = batch
        users = interaction[config['USER_ID_FIELD']].numpy()
        pu, pi = positive_u.numpy(), positive_i.numpy()
        for lu, x in zip(pu, pi):
            pos_item[int(users[lu])] = int(x)
        if history_index is not None and len(history_index)==2:
            rows, cols = history_index[0].numpy(), history_index[1].numpy()
            for r,c in zip(rows,cols):
                mask.setdefault(int(users[r]), set()).add(int(c))
    return pos_item, mask

test_item, test_mask = extract_split(test_data)
valid_item, valid_mask = extract_split(valid_data)
print(f"Test: {len(test_item):,} | Valid: {len(valid_item):,}")

# historique d'entraînement de chaque user (tout sauf valid+test)
train_pos = defaultdict(set)
inter_all = dataset.inter_feat
uu = inter_all[config['USER_ID_FIELD']].numpy()
ii = inter_all[config['ITEM_ID_FIELD']].numpy()
for u,i in zip(uu,ii):
    train_pos[int(u)].add(int(i))
for u in list(train_pos.keys()):
    train_pos[u].discard(test_item.get(u,-1))
    train_pos[u].discard(valid_item.get(u,-1))
print("Historique train construit pour", len(train_pos), "users")

Test: 53,897 | Valid: 53,897
Historique train construit pour 53897 users


## Étape 2 — Embeddings pré-entraînés (initialisation)

On charge les embeddings **LightGCN** (propagés) et **RotatE v2**, alignés sur les index RecBole. Ils serviront à **initialiser** les paramètres entraînables du modèle (fine-tuning).

In [5]:
# --- LightGCN : embeddings propagés ---
m = get_model('LightGCN')(config, train_data._dataset).to(DEVICE)
ck = torch.load(LGCN_PATH, map_location=DEVICE)
m.load_state_dict(ck['state_dict']); m.eval(); m = m.cpu()
if hasattr(m,'norm_adj_matrix'): m.norm_adj_matrix = m.norm_adj_matrix.cpu()
if hasattr(m,'restore_user_e'): m.restore_user_e=None; m.restore_item_e=None
with torch.no_grad():
    ue_cf, ie_cf = m.forward()
    ue_cf = ue_cf.numpy().astype(np.float32)   # [n_users,64]
    ie_cf = ie_cf.numpy().astype(np.float32)   # [n_items,64]
print("LightGCN emb:", ue_cf.shape, ie_cf.shape)

c:\Users\AMIRA\anaconda3\envs\llmkger\lib\site-packages\recbole\model\general_recommender\lightgcn.py:125: UserWarning: torch.sparse.SparseTensor(indices, values, shape, *, device=) is deprecated.  Please use torch.sparse_coo_tensor(indices, values, shape, dtype=, device=). (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_new.cpp:646.)
  SparseL = torch.sparse.FloatTensor(i, data, torch.Size(L.shape))


LightGCN emb: (53898, 64) (2813, 64)


In [6]:
# --- RotatE v2 : embeddings entités (complexe -> réel) ---
rotate = torch.load(os.path.join(KGE_DIR,'trained_model.pkl'), map_location='cpu')
rotate.eval()
ent_df = pd.read_csv(os.path.join(KGE_DIR,'training_triples','entity_to_id.tsv.gz'),sep='\t',compression='gzip')
kg_entity2id = dict(zip(ent_df['label'].astype(str), ent_df['id']))

# récupérer la matrice d'embeddings entités de PyKEEN
ent_emb = rotate.entity_representations[0]().detach().cpu().numpy()  # complexe -> [N, dim] (peut être complexe)
print("RotatE emb brut:", ent_emb.shape, ent_emb.dtype)
# si complexe : concatener partie reelle + imaginaire
if np.iscomplexobj(ent_emb):
    ent_emb = np.concatenate([ent_emb.real, ent_emb.imag], axis=1).astype(np.float32)
else:
    ent_emb = ent_emb.astype(np.float32)
print("RotatE emb (réel):", ent_emb.shape)
KG_DIM = ent_emb.shape[1]

RotatE emb brut: (235055, 128) complex64
RotatE emb (réel): (235055, 256)


In [7]:
# Aligner RotatE sur les index RecBole (users et items)
def build_kg_init(id2token, n, kg_entity2id, ent_emb, dim):
    M = np.zeros((n, dim), dtype=np.float32)
    found = 0
    for idx in range(n):
        tok = id2token.get(idx)
        if tok is None: continue
        eid = kg_entity2id.get(str(tok))
        if eid is not None:
            M[idx] = ent_emb[eid]; found += 1
    return M, found

user_kg_init, fu = build_kg_init(id2user, n_users, kg_entity2id, ent_emb, KG_DIM)
item_kg_init, fi = build_kg_init(id2item, n_items, kg_entity2id, ent_emb, KG_DIM)
print(f"KG init — users trouvés: {fu:,}/{n_users} | items trouvés: {fi:,}/{n_items}")
print(f"Dimension KG: {KG_DIM}")

KG init — users trouvés: 52,020/53898 | items trouvés: 2,812/2813
Dimension KG: 256


## Étape 3 — P_prereq (signal fixe)

P_prereq reste un signal **externe fixe** (calculé depuis le KG). On construit une fonction qui donne P_prereq(u, i) pour n'importe quel couple, à partir des structures KG (comme en Phase 5).

In [8]:
# structures prérequis (identiques à extract_pprereq)
kg = pd.read_csv(os.path.join(KG_DIR,'llmkger.kg'), sep='\t'); kg.columns=['head','relation','tail']
concepts_of_course = defaultdict(set)
for h,t in kg[kg['relation']=='belongs_to'][['head','tail']].itertuples(index=False):
    h,t=str(h),str(t)
    if h.startswith('K') and t.startswith('C_'): concepts_of_course[t].add(h)
    elif t.startswith('K') and h.startswith('C_'): concepts_of_course[h].add(t)
prereqs_of_concept = defaultdict(set)
for h,t in kg[kg['relation']=='prerequisite_of'][['head','tail']].itertuples(index=False):
    h,t=str(h),str(t)
    if h.startswith('K') and t.startswith('K'): prereqs_of_concept[t].add(h)
prereq_of_course = {}
for course,concs in concepts_of_course.items():
    req=set()
    for c in concs: req|=prereqs_of_concept.get(c,set())
    if req: prereq_of_course[course]=req

# concepts couverts par user (via historique train) -- en index interne
concepts_internal = {item2id.get(c,0):concs for c,concs in concepts_of_course.items() if item2id.get(c,0)>0}
cov_cache = {}
def covered(u_idx):
    if u_idx in cov_cache: return cov_cache[u_idx]
    cov=set()
    for ci in train_pos.get(u_idx,()):
        cov|=concepts_internal.get(ci,set())
    cov_cache[u_idx]=cov; return cov

# P_prereq(u_idx, i_idx)
def pprereq(u_idx, i_idx):
    course = id2item.get(i_idx)
    req = prereq_of_course.get(course)
    if not req: return 0.0
    cov = covered(u_idx)
    return len(req-cov)/len(req)
print("P_prereq prêt. Cours avec prérequis:", len(prereq_of_course))

P_prereq prêt. Cours avec prérequis: 584


In [9]:
# REMPLACEMENT du pré-calcul long : P_prereq à la volée par batch
# On ne précalcule rien — on calcule uniquement les paires nécessaires
# pendant l'entraînement (batch par batch)

P_lookup = {}   # on laisse vide — non utilisé dans la nouvelle version

def p_batch_fast(u_arr, i_arr):
    """
    Calcule P_prereq pour un batch de paires (u, i).
    Seulement les cours avec prérequis sont traités (les autres = 0).
    """
    result = np.zeros(len(u_arr), dtype=np.float32)
    for k in range(len(u_arr)):
        u_idx = int(u_arr[k])
        i_idx = int(i_arr[k])
        course = id2item.get(i_idx)
        req = prereq_of_course.get(course)
        if not req:
            continue
        cov = covered(u_idx)
        nc = len(req - cov)
        if nc > 0:
            result[k] = nc / len(req)
    return torch.tensor(result, dtype=torch.float32, device=DEVICE)

# Test rapide sur 5 paires
import random
test_u = [random.randint(1, n_users-1) for _ in range(5)]
test_i = random.choices(candidate_items, k=5)
ptest  = p_batch_fast(np.array(test_u), np.array(test_i))
print("Test p_batch_fast (5 paires):", ptest.tolist())
print("✅ P_prereq à la volée prêt — aucun pré-calcul nécessaire")

Test p_batch_fast (5 paires): [0.800000011920929, 0.0, 0.0, 1.0, 0.0]
✅ P_prereq à la volée prêt — aucun pré-calcul nécessaire


## Étape 4 — Le modèle end-to-end

Paramètres **entraînables** : embeddings KG (init RotatE) + collaboratifs (init LightGCN) + poids de fusion (α, β, λ init à tes valeurs Phase 5).

Score(u,i) = α·⟨u_kg, i_kg⟩ + β·⟨u_cf, i_cf⟩ − λ·P_prereq(u,i)

In [10]:
# Paires d'entraînement
train_pairs = [(u,i) for u,items in train_pos.items() for i in items]
train_pairs = np.array(train_pairs, dtype=np.int64)
print(f"Paires d'entraînement : {len(train_pairs):,}")

cand_arr = np.array(candidate_items, dtype=np.int64)
cand_set_of_user = {u:set(items) for u,items in train_pos.items()}

def sample_neg(u):
    while True:
        j = int(cand_arr[np.random.randint(len(cand_arr))])
        if j not in cand_set_of_user.get(u,()): return j

print("Paires et négatifs prêts ✅")

Paires d'entraînement : 1,876,701
Paires et négatifs prêts ✅


In [11]:
import torch.nn as nn
import torch.nn.functional as F

class LLMKGERe2e_v3(nn.Module):
    def __init__(self, user_kg, item_kg, user_cf, item_cf, a=0.607, b=0.334, l=0.059):
        super().__init__()
        KG_RAW_DIM = user_kg.shape[1]
        CF_DIM     = user_cf.shape[1]
        HIDDEN_DIM = CF_DIM

        self.user_kg = nn.Embedding.from_pretrained(torch.tensor(user_kg), freeze=False)
        self.item_kg = nn.Embedding.from_pretrained(torch.tensor(item_kg), freeze=False)
        self.user_cf = nn.Embedding.from_pretrained(torch.tensor(user_cf), freeze=False)
        self.item_cf = nn.Embedding.from_pretrained(torch.tensor(item_cf), freeze=False)

        self.kg_proj = nn.Linear(KG_RAW_DIM, HIDDEN_DIM, bias=False)
        nn.init.xavier_uniform_(self.kg_proj.weight)

        # Normalisation par statistiques FIXES (buffers, pas BatchNorm dynamique)
        # seront calculées après création via calibrate()
        self.register_buffer('kg_mean', torch.tensor(0.0))
        self.register_buffer('kg_std',  torch.tensor(1.0))
        self.register_buffer('emb_mean', torch.tensor(0.0))
        self.register_buffer('emb_std',  torch.tensor(1.0))

        init = torch.log(torch.tensor([a, b, l]) + 1e-6)
        self.weight_logits = nn.Parameter(init)

    def _weights(self):
        w = F.softmax(self.weight_logits, dim=0)
        return w[0], w[1], w[2]

    def s_kg_raw(self, u, i):
        pu = self.kg_proj(self.user_kg(u))
        pi = self.kg_proj(self.item_kg(i))
        return (pu * pi).sum(-1)

    def s_emb_raw(self, u, i):
        return (self.user_cf(u) * self.item_cf(i)).sum(-1)

    def score(self, u, i, p):
        skg  = (self.s_kg_raw(u, i)  - self.kg_mean)  / (self.kg_std  + 1e-6)
        semb = (self.s_emb_raw(u, i) - self.emb_mean) / (self.emb_std + 1e-6)
        a, b, l = self._weights()
        return a*skg + b*semb - l*p

model = LLMKGERe2e_v3(user_kg_init, item_kg_init, ue_cf, ie_cf).to(DEVICE)

# CALIBRATION : calculer les stats fixes de S_KG et S_emb sur un échantillon
model.eval()
with torch.no_grad():
    sample_idx = np.random.choice(len(train_pairs), 20000, replace=False)
    su = torch.tensor(train_pairs[sample_idx,0], device=DEVICE)
    si = torch.tensor(train_pairs[sample_idx,1], device=DEVICE)
    skg_vals  = model.s_kg_raw(su, si)
    semb_vals = model.s_emb_raw(su, si)
    model.kg_mean  = skg_vals.mean(); model.kg_std  = skg_vals.std()
    model.emb_mean = semb_vals.mean(); model.emb_std = semb_vals.std()

a0,b0,l0 = model._weights()
print(f"Modèle v3 créé et calibré.")
print(f"  S_KG  : mean={model.kg_mean.item():.3f}, std={model.kg_std.item():.3f}")
print(f"  S_emb : mean={model.emb_mean.item():.3f}, std={model.emb_std.item():.3f}")
print(f"  Poids initiaux : α={a0.item():.3f} β={b0.item():.3f} λ={l0.item():.3f}")

Modèle v3 créé et calibré.
  S_KG  : mean=-7.585, std=4.561
  S_emb : mean=4.769, std=1.565
  Poids initiaux : α=0.607 β=0.334 λ=0.059


In [12]:
# ===== Ancre L_KG : sauvegarder les embeddings KG initiaux (RotatE projeté) =====
with torch.no_grad():
    anchor_user_kg = model.user_kg.weight.detach().clone()
    anchor_item_kg = model.item_kg.weight.detach().clone()
print(f"Ancre L_KG sauvegardée :")
print(f"  user_kg : {anchor_user_kg.shape}")
print(f"  item_kg : {anchor_item_kg.shape}")
print("Ces valeurs (RotatE initial) servent de référence pour empêcher la dérive.")

Ancre L_KG sauvegardée :
  user_kg : torch.Size([53898, 256])
  item_kg : torch.Size([2813, 256])
Ces valeurs (RotatE initial) servent de référence pour empêcher la dérive.


In [16]:
opt = torch.optim.Adam(model.parameters(), lr=0.001)
EPOCHS = 15; BATCH = 2048
GAMMA = 0.1   # force de l'ancrage KG (à ajuster si besoin)
n = len(train_pairs)
history_loss = []

print(f"Entraînement multi-tâche — L = L_rec + {GAMMA}·L_KG")
print(f"{'Ep':>4} | {'L_total':>9} | {'L_rec':>9} | {'L_KG':>9} | {'α':>6} {'β':>6} {'λ':>6} | {'t':>6}")
print("-"*72)

for ep in range(1, EPOCHS+1):
    model.train()
    perm = np.random.permutation(n)
    tot=0.0; tot_rec=0.0; tot_kg=0.0; nb=0; t0=time.time()
    for s in range(0, n, BATCH):
        idx = perm[s:s+BATCH]
        u    = train_pairs[idx, 0]
        ipos = train_pairs[idx, 1]
        ineg = np.array([sample_neg(uu) for uu in u], dtype=np.int64)

        ut   = torch.tensor(u,    device=DEVICE)
        ipt  = torch.tensor(ipos, device=DEVICE)
        int_ = torch.tensor(ineg, device=DEVICE)

        p_pos = p_batch_fast(u, ipos)
        p_neg = p_batch_fast(u, ineg)

        # --- L_rec (BPR) ---
        spos = model.score(ut, ipt, p_pos)
        sneg = model.score(ut, int_, p_neg)
        L_rec = -torch.log(torch.sigmoid(spos - sneg) + 1e-9).mean()

        # --- L_KG (ancrage : ne pas dériver de RotatE) ---
        du_pos = model.user_kg(ut)  - anchor_user_kg[ut]
        di_pos = model.item_kg(ipt) - anchor_item_kg[ipt]
        di_neg = model.item_kg(int_) - anchor_item_kg[int_]
        L_KG = (du_pos.pow(2).sum(-1).mean()
                + di_pos.pow(2).sum(-1).mean()
                + di_neg.pow(2).sum(-1).mean()) / 3.0

        # --- Loss totale multi-tâche ---
        loss = L_rec + GAMMA * L_KG

        opt.zero_grad(); loss.backward(); opt.step()
        tot += loss.item(); tot_rec += L_rec.item(); tot_kg += L_KG.item(); nb += 1

    a,b,l = model._weights()
    history_loss.append(tot/nb)
    print(f"{ep:>4} | {tot/nb:>9.4f} | {tot_rec/nb:>9.4f} | {tot_kg/nb:>9.4f} | "
          f"{a.item():>6.3f} {b.item():>6.3f} {l.item():>6.3f} | {time.time()-t0:>5.0f}s")

print("\n✅ Entraînement multi-tâche terminé")

Entraînement multi-tâche — L = L_rec + 0.1·L_KG
  Ep |   L_total |     L_rec |      L_KG |      α      β      λ |      t
------------------------------------------------------------------------
   1 |    0.1502 |    0.1465 |    0.0370 |  0.343  0.625  0.032 |   624s
   2 |    0.1144 |    0.1130 |    0.0141 |  0.204  0.777  0.018 |   936s
   3 |    0.1050 |    0.1043 |    0.0078 |  0.135  0.853  0.012 |   882s
   4 |    0.1007 |    0.1002 |    0.0059 |  0.093  0.899  0.008 |   900s
   5 |    0.0978 |    0.0972 |    0.0052 |  0.067  0.928  0.006 |   826s
   6 |    0.0950 |    0.0945 |    0.0051 |  0.049  0.947  0.004 |   502s
   7 |    0.0931 |    0.0926 |    0.0050 |  0.037  0.960  0.003 |   454s
   8 |    0.0906 |    0.0901 |    0.0050 |  0.030  0.968  0.002 |   526s
   9 |    0.0893 |    0.0888 |    0.0053 |  0.026  0.973  0.001 |   507s


KeyboardInterrupt: 

In [13]:
import numpy as np, torch

print("Extraction des embeddings affinés (post end-to-end)...")
model.eval()
with torch.no_grad():
    U_kg = model.kg_proj(model.user_kg.weight.detach())
    I_kg = model.kg_proj(model.item_kg.weight.detach())
    U_cf = model.user_cf.weight.detach()
    I_cf = model.item_cf.weight.detach()
    kgm, kgs = model.kg_mean.item(), model.kg_std.item()
    em, es   = model.emb_mean.item(), model.emb_std.item()

cand_arr = np.array(candidate_items, dtype=np.int64)
M = len(cand_arr)
N = len(test_item)

test_users = list(test_item.keys())
test_u_idx = np.array(test_users, dtype=np.int64)

Ikg_cand = I_kg[cand_arr]
Icf_cand = I_cf[cand_arr]

print("Calcul matrice S_KG affinée...")
mat_skg = np.zeros((N, M), dtype=np.float32)
for s in range(0, N, 2000):
    u = test_u_idx[s:s+2000]
    ut = torch.tensor(u, device=DEVICE)
    scores = (U_kg[ut] @ Ikg_cand.T).cpu().numpy()
    mat_skg[s:s+2000] = (scores - kgm) / (kgs + 1e-6)
print("  S_KG:", mat_skg.shape)

print("Calcul matrice S_emb affinée...")
mat_semb = np.zeros((N, M), dtype=np.float32)
for s in range(0, N, 2000):
    u = test_u_idx[s:s+2000]
    ut = torch.tensor(u, device=DEVICE)
    scores = (U_cf[ut] @ Icf_cand.T).cpu().numpy()
    mat_semb[s:s+2000] = (scores - em) / (es + 1e-6)
print("  S_emb:", mat_semb.shape)

print("✅ Matrices S_KG et S_emb affinées calculées")

Extraction des embeddings affinés (post end-to-end)...
Calcul matrice S_KG affinée...
  S_KG: (53897, 2812)
Calcul matrice S_emb affinée...
  S_emb: (53897, 2812)
✅ Matrices S_KG et S_emb affinées calculées


In [14]:
import numpy as np

# Index de colonne pour chaque cours candidat
cand_index = {it:j for j,it in enumerate(candidate_items)}

# ===== Réutiliser P_prereq déjà calculée (Phase 5) =====
print("Rechargement de la matrice P_prereq (Phase 5)...")
mat_p_phase5 = np.load(os.path.join(RESULTS_REC, 'mat_P_prereq_norm.npy'))
print("  P_prereq Phase 5:", mat_p_phase5.shape)

# MAIS attention : la matrice Phase 5 est indexée par l'ordre de S_emb_scores_final.csv
# alors que nos matrices actuelles sont indexées par test_u_idx (ordre du split RecBole)
# Il faut vérifier l'alignement.

# Charger l'ordre de référence Phase 5
df_ref = pd.read_csv(os.path.join(RESULTS_REC, 'S_emb_scores_final.csv'))
ref_users = df_ref['user_id'].astype(str).tolist()
ref_user2row = {u:i for i,u in enumerate(ref_users)}

# Aligner sur l'ordre de test_u_idx
print("Alignement sur l'ordre test_u_idx...")
mat_p = np.zeros((N, M), dtype=np.float32)
n_found = 0
for row, u_idx in enumerate(test_u_idx):
    u_ext = id2user.get(int(u_idx), None)
    if u_ext is None: continue
    phase5_row = ref_user2row.get(str(u_ext))
    if phase5_row is not None:
        mat_p[row] = mat_p_phase5[phase5_row]
        n_found += 1
print(f"  Users alignés : {n_found:,}/{N}")
print(f"  P_prereq mean : {mat_p.mean():.4f}")

# ===== Colonnes cibles + masques (TEST et VALID) =====
test_col = np.array([cand_index.get(test_item.get(int(u),-1),-1) for u in test_u_idx], dtype=np.int64)
print(f"Cours test mappés : {(test_col>=0).sum():,}/{N}")

mask_test = np.zeros((N, M), dtype=bool)
mask_valid = np.zeros((N, M), dtype=bool)
valid_col = np.full(N, -1, dtype=np.int64)

for row, u in enumerate(test_u_idx):
    u = int(u)
    for c in test_mask.get(u, ()):
        jm = cand_index.get(c)
        if jm is not None: mask_test[row, jm] = True
    vc = valid_item.get(u, -1)
    valid_col[row] = cand_index.get(vc, -1)
    for c in valid_mask.get(u, ()):
        jm = cand_index.get(c)
        if jm is not None: mask_valid[row, jm] = True

print(f"Cours valid mappés : {(valid_col>=0).sum():,}/{N}")
print("✅ P_prereq rechargée, masques prêts — instantané !")

Rechargement de la matrice P_prereq (Phase 5)...
  P_prereq Phase 5: (53897, 2812)
Alignement sur l'ordre test_u_idx...
  Users alignés : 53,897/53897
  P_prereq mean : 0.1860
Cours test mappés : 53,897/53897
Cours valid mappés : 53,897/53897
✅ P_prereq rechargée, masques prêts — instantané !


In [21]:
import numpy as np, math

# Vérification 1 : vérifier que le cours TEST est bien classé par S_emb seul
# (on devrait retrouver ~0.0986 comme en Phase 5)
print("=== Diagnostic alignement ===\n")

# Test S_emb seul sur le TEST (pas le valid)
score_test = mat_semb.copy().astype(np.float64)
score_test[mask_test] = -1e18

ndcg_emb_test = 0.0; v = 0; ranks = []
for i in range(N):
    j = test_col[i]
    if j < 0: continue
    target = score_test[i, j]
    greater = int((score_test[i] > target).sum())
    ties = int((score_test[i] == target).sum())
    rank = greater + (ties+1)/2.0
    ranks.append(rank)
    if rank <= 10:
        ndcg_emb_test += 1.0/math.log2(rank+1)
    v += 1

print(f"S_emb seul sur TEST : NDCG@10 = {ndcg_emb_test/v:.4f}")
print(f"  (Phase 5 donnait 0.0986 — si très différent, problème d'alignement)")
print(f"  Rang médian : {np.median(ranks):.0f} / {M}")

# Test S_emb seul sur le VALID
score_valid = mat_semb.copy().astype(np.float64)
score_valid[mask_valid] = -1e18

ndcg_emb_valid = 0.0; v2 = 0; ranks_v = []
for i in range(N):
    j = valid_col[i]
    if j < 0: continue
    target = score_valid[i, j]
    greater = int((score_valid[i] > target).sum())
    ties = int((score_valid[i] == target).sum())
    rank = greater + (ties+1)/2.0
    ranks_v.append(rank)
    if rank <= 10:
        ndcg_emb_valid += 1.0/math.log2(rank+1)
    v2 += 1

print(f"\nS_emb seul sur VALID : NDCG@10 = {ndcg_emb_valid/v2:.4f}")
print(f"  Rang médian valid : {np.median(ranks_v):.0f} / {M}")

# Vérif : le cours test du user 0 est bien dans quelle colonne ?
i = 0
print(f"\nUser 0 (idx={test_u_idx[0]}) :")
print(f"  test_col[0] = {test_col[0]}")
print(f"  valid_col[0] = {valid_col[0]}")
print(f"  score test   : {mat_semb[0, test_col[0]]:.4f}")
print(f"  score valid  : {mat_semb[0, valid_col[0]]:.4f}")
print(f"  Combien de scores > test  : {(mat_semb[0] > mat_semb[0,test_col[0]]).sum()}")
print(f"  Combien de scores > valid : {(mat_semb[0] > mat_semb[0,valid_col[0]]).sum()}")

=== Diagnostic alignement ===

S_emb seul sur TEST : NDCG@10 = 0.0834
  (Phase 5 donnait 0.0986 — si très différent, problème d'alignement)
  Rang médian : 96 / 2812

S_emb seul sur VALID : NDCG@10 = 0.0820
  Rang médian valid : 94 / 2812

User 0 (idx=1) :
  test_col[0] = 1
  valid_col[0] = 14
  score test   : 0.5737
  score valid  : 0.6717
  Combien de scores > test  : 91
  Combien de scores > valid : 84


In [22]:
from skopt import gp_minimize
from skopt.space import Real
import time, math

# NDCG@10 sur la VALIDATION avec les embeddings affinés
def ndcg_valid_b1(alpha, beta, lam):
    score = (alpha * mat_skg + beta * mat_semb - lam * mat_p).astype(np.float64)
    score[mask_valid] = -1e18
    s = 0.0; v = 0
    for i in range(N):
        j = valid_col[i]
        if j < 0: continue
        target = score[i, j]
        greater = int((score[i] > target).sum())
        ties    = int((score[i] == target).sum())
        rank    = greater + (ties + 1) / 2.0
        if rank <= 10:
            s += 1.0 / math.log2(rank + 1)
        v += 1
    return s / v if v else 0.0

# Tester quelques combinaisons pour cadrer l'espace
print("Sondage initial des poids :")
for a,b,l in [(0.0,1.0,0.0),(1.0,0.0,0.0),(0.0,0.0,1.0),(0.5,0.5,0.0),(0.3,0.6,0.1)]:
    v = ndcg_valid_b1(a,b,l)
    print(f"  α={a:.1f} β={b:.1f} λ={l:.1f} → NDCG@10 valid = {v:.4f}")

print("\nLancement Bayesian Optimization sur la validation...")
space = [Real(0.0, 1.0, name='alpha'), Real(0.0, 1.0, name='beta')]

def objective(params):
    a, b = params
    if a + b > 1.0: return 0.0
    l = 1.0 - a - b
    return -ndcg_valid_b1(a, b, l)

t0 = time.time()
res = gp_minimize(objective, space, n_calls=40, n_initial_points=12,
                  random_state=42, verbose=False)
print(f"Terminé en {(time.time()-t0)/60:.1f} min")

a_opt, b_opt = res.x
l_opt = 1.0 - a_opt - b_opt
ndcg_v = -res.fun
print(f"\n🏆 Poids optimaux (sur validation, embeddings affinés) :")
print(f"   α (S_KG)    = {a_opt:.4f}")
print(f"   β (S_emb)   = {b_opt:.4f}")
print(f"   λ (P_prereq)= {l_opt:.4f}")
print(f"   NDCG@10 valid = {ndcg_v:.4f}")
print(f"\n   Phase 5 valid : 0.1027")
print(f"   Différence    : {ndcg_v - 0.1027:+.4f}")

Sondage initial des poids :
  α=0.0 β=1.0 λ=0.0 → NDCG@10 valid = 0.0820
  α=1.0 β=0.0 λ=0.0 → NDCG@10 valid = 0.0175
  α=0.0 β=0.0 λ=1.0 → NDCG@10 valid = 0.0000
  α=0.5 β=0.5 λ=0.0 → NDCG@10 valid = 0.0517
  α=0.3 β=0.6 λ=0.1 → NDCG@10 valid = 0.0698

Lancement Bayesian Optimization sur la validation...
Terminé en 4.0 min

🏆 Poids optimaux (sur validation, embeddings affinés) :
   α (S_KG)    = 0.0263
   β (S_emb)   = 0.3053
   λ (P_prereq)= 0.6683
   NDCG@10 valid = 0.0955

   Phase 5 valid : 0.1027
   Différence    : -0.0072


In [23]:
# Contrôle : vérifier que les ex-aequo ne faussent pas le résultat
score_check = (0.0263 * mat_skg + 0.3053 * mat_semb - 0.6683 * mat_p).astype(np.float64)
score_check[mask_valid] = -1e18

ties_counts = []
for i in range(0, N, 100):  # échantillon
    j = valid_col[i]
    if j < 0: continue
    target = score_check[i, j]
    ties = int((score_check[i] == target).sum())
    ties_counts.append(ties)

import numpy as np
print(f"Ex-aequo médian avec le cours valid : {np.median(ties_counts):.0f}")
print(f"Ex-aequo max : {max(ties_counts)}")
print(f"\nSi ex-aequo médian > 100 → artefact de ties (même bug que Phase 5)")
print(f"Si ex-aequo médian < 10  → résultat honnête")

Ex-aequo médian avec le cours valid : 1
Ex-aequo max : 1

Si ex-aequo médian > 100 → artefact de ties (même bug que Phase 5)
Si ex-aequo médian < 10  → résultat honnête


In [24]:
import math, numpy as np

def ndcg_test_b1(alpha, beta, lam, ks=(5,10)):
    score = (alpha * mat_skg + beta * mat_semb - lam * mat_p).astype(np.float64)
    score[mask_test] = -1e18
    res = {f'{m}@{k}':[] for k in ks for m in ['ndcg','mrr','recall']}
    for i in range(N):
        j = test_col[i]
        if j < 0: continue
        target = score[i, j]
        greater = int((score[i] > target).sum())
        ties    = int((score[i] == target).sum())
        rank    = greater + (ties + 1) / 2.0
        for k in ks:
            hit = 1.0 if rank <= k else 0.0
            res[f'recall@{k}'].append(hit)
            res[f'ndcg@{k}'].append(1.0/math.log2(rank+1) if rank<=k else 0.0)
            res[f'mrr@{k}'].append(1.0/rank if rank<=k else 0.0)
    return {m: float(np.mean(v)) for m,v in res.items()}

print("Évaluation finale sur le TEST (B1 : embeddings affinés + poids optimaux)...")
res_b1 = ndcg_test_b1(a_opt, b_opt, l_opt)

print("\n" + "="*60)
print("  COMPARAISON FINALE")
print("="*60)
print(f"{'Métrique':<14} {'Pipeline':>10} {'End-to-end B1':>14} {'Δ':>8}")
print("-"*60)
pipeline = {'ndcg@10':0.1037,'mrr@10':0.0919,'recall@10':0.1427,
            'ndcg@5':0.0940,'mrr@5':0.0880,'recall@5':0.1124}
for k in ['ndcg@10','mrr@10','recall@10','ndcg@5','mrr@5','recall@5']:
    p = pipeline[k]; e = res_b1[k]
    print(f"{k:<14} {p:>10.4f} {e:>14.4f} {e-p:>+8.4f}")
print("="*60)
print(f"\nPoids B1 : α={a_opt:.3f} β={b_opt:.3f} λ={l_opt:.3f}")
print(f"Valid B1 : {-res.fun:.4f}  |  Phase 5 valid : 0.1027")

Évaluation finale sur le TEST (B1 : embeddings affinés + poids optimaux)...

  COMPARAISON FINALE
Métrique         Pipeline  End-to-end B1        Δ
------------------------------------------------------------
ndcg@10            0.1037         0.0971  -0.0066
mrr@10             0.0919         0.0748  -0.0171
recall@10          0.1427         0.1706  +0.0279
ndcg@5             0.0940         0.0798  -0.0142
mrr@5              0.0880         0.0677  -0.0203
recall@5           0.1124         0.1167  +0.0043

Poids B1 : α=0.026 β=0.305 λ=0.668
Valid B1 : 0.0955  |  Phase 5 valid : 0.1027


In [25]:
import torch, os, json

# Sauvegarder le modèle end-to-end (embeddings affinés)
save_dir = os.path.join(RESULTS_REC, 'e2e_v3')
os.makedirs(save_dir, exist_ok=True)

torch.save(model.state_dict(), os.path.join(save_dir, 'llmkger_e2e_v3.pt'))

# Sauvegarder les résultats
results_l1 = {
    'pipeline'     : {'ndcg@10':0.1037, 'mrr@10':0.0919, 'recall@10':0.1427},
    'e2e_v3'       : {'ndcg@10':0.0885, 'mrr@10':0.0667, 'recall@10':0.1603},
    'e2e_b1'       : {'ndcg@10':0.0971, 'mrr@10':0.0748, 'recall@10':0.1706,
                      'alpha':a_opt, 'beta':b_opt, 'lambda':l_opt},
    'note': 'Limitation 1 corrigée : architecture end-to-end non figée, '
            'loss commune L_rec + gamma*L_KG. '
            'Base pour les améliorations suivantes (L2 attention, L4 prérequis).'
}

with open(os.path.join(save_dir, 'results_L1.json'), 'w') as f:
    json.dump(results_l1, f, indent=2)

print("💾 Modèle end-to-end et résultats L1 sauvegardés")
print(f"   Dossier : {save_dir}")
print("\n=== PROGRESSION DU PROJET ===")
print(f"Pipeline original  : NDCG@10=0.1037 (référence, figé)")
print(f"End-to-end L1      : NDCG@10=0.0971 (non figé ✅, base pour L2+)")
print(f"\nProchaine étape : Limitation 2 — Attention-based fusion")
print(f"Objectif : remplacer α·S_KG + β·S_emb - λ·P_prereq (poids fixes)")
print(f"        par une fusion adaptative (poids différents par utilisateur)")

💾 Modèle end-to-end et résultats L1 sauvegardés
   Dossier : D:\Memoire\LLMKGER\reco_results_final\e2e_v3

=== PROGRESSION DU PROJET ===
Pipeline original  : NDCG@10=0.1037 (référence, figé)
End-to-end L1      : NDCG@10=0.0971 (non figé ✅, base pour L2+)

Prochaine étape : Limitation 2 — Attention-based fusion
Objectif : remplacer α·S_KG + β·S_emb - λ·P_prereq (poids fixes)
        par une fusion adaptative (poids différents par utilisateur)


In [26]:
import os, torch

# Vérifier que le modèle v3 est bien sauvegardé
save_dir = os.path.join(RESULTS_REC, 'e2e_v3')
model_path = os.path.join(save_dir, 'llmkger_e2e_v3.pt')

if os.path.exists(model_path):
    sz = os.path.getsize(model_path)/1e6
    print(f"✅ Modèle v3 trouvé : {model_path} ({sz:.1f} Mo)")
else:
    print(f"❌ Modèle v3 introuvable : {model_path}")
    print("Il faudra le réentraîner.")

# Vérifier aussi les matrices affinées
for name, var in [('mat_skg', 'mat_skg'), ('mat_semb', 'mat_semb'), ('mat_p', 'mat_p')]:
    try:
        m = eval(var)
        print(f"✅ {name} en mémoire : {m.shape}")
    except:
        print(f"⚠️  {name} pas en mémoire (normal si PC redémarré)")

✅ Modèle v3 trouvé : D:\Memoire\LLMKGER\reco_results_final\e2e_v3\llmkger_e2e_v3.pt (72.7 Mo)
✅ mat_skg en mémoire : (53897, 2812)
✅ mat_semb en mémoire : (53897, 2812)
✅ mat_p en mémoire : (53897, 2812)


In [14]:
import torch, os
import torch.nn as nn
import torch.nn.functional as F

# Redéfinir la classe (nécessaire pour charger le state_dict)
class LLMKGERe2e_v3(nn.Module):
    def __init__(self, n_users, n_items, kg_raw_dim=256, cf_dim=64):
        super().__init__()
        self.user_kg = nn.Embedding(n_users, kg_raw_dim)
        self.item_kg = nn.Embedding(n_items, kg_raw_dim)
        self.user_cf = nn.Embedding(n_users, cf_dim)
        self.item_cf = nn.Embedding(n_items, cf_dim)
        self.kg_proj = nn.Linear(kg_raw_dim, cf_dim, bias=False)
        self.register_buffer('kg_mean',  torch.tensor(0.0))
        self.register_buffer('kg_std',   torch.tensor(1.0))
        self.register_buffer('emb_mean', torch.tensor(0.0))
        self.register_buffer('emb_std',  torch.tensor(1.0))
        self.weight_logits = nn.Parameter(torch.zeros(3))

    def _weights(self):
        w = F.softmax(self.weight_logits, dim=0)
        return w[0], w[1], w[2]

    def s_kg_raw(self, u, i):
        pu = self.kg_proj(self.user_kg(u))
        pi = self.kg_proj(self.item_kg(i))
        return (pu * pi).sum(-1)

    def s_emb_raw(self, u, i):
        return (self.user_cf(u) * self.item_cf(i)).sum(-1)

    def score(self, u, i, p):
        skg  = (self.s_kg_raw(u, i)  - self.kg_mean)  / (self.kg_std  + 1e-6)
        semb = (self.s_emb_raw(u, i) - self.emb_mean) / (self.emb_std + 1e-6)
        a, b, l = self._weights()
        return a*skg + b*semb - l*p

# Charger le modèle depuis le disque
model = LLMKGERe2e_v3(n_users=53898, n_items=2813).to(DEVICE)
ck = torch.load(
    os.path.join(RESULTS_REC, 'e2e_v3', 'llmkger_e2e_v3.pt'),
    map_location=DEVICE)
model.load_state_dict(ck)
model.eval()

a, b, l = model._weights()
print(f"✅ Modèle v3 chargé depuis le disque")
print(f"   kg_mean={model.kg_mean.item():.3f}  kg_std={model.kg_std.item():.3f}")
print(f"   emb_mean={model.emb_mean.item():.3f}  emb_std={model.emb_std.item():.3f}")
print(f"   Poids : α={a.item():.3f} β={b.item():.3f} λ={l.item():.3f}")

✅ Modèle v3 chargé depuis le disque
   kg_mean=-6.692  kg_std=4.736
   emb_mean=4.769  emb_std=1.565
   Poids : α=0.025 β=0.973 λ=0.001


In [15]:
# Vérifier que tout est en mémoire depuis le notebook EndToEnd
print("=== Vérification des prérequis ===")
checks = {
    'model (v3)': 'model',
    'mat_skg': 'mat_skg',
    'mat_semb': 'mat_semb',
    'mat_p': 'mat_p',
    'test_col': 'test_col',
    'valid_col': 'valid_col',
    'mask_test': 'mask_test',
    'mask_valid': 'mask_valid',
    'train_pairs': 'train_pairs',
    'test_u_idx': 'test_u_idx',
}
all_ok = True
for name, var in checks.items():
    try:
        v = eval(var)
        shape = getattr(v, 'shape', len(v) if hasattr(v,'__len__') else 'ok')
        print(f"  ✅ {name}: {shape}")
    except:
        print(f"  ❌ {name}: MANQUANT")
        all_ok = False
print("\n✅ Tout prêt — on peut construire la fusion attention" if all_ok else
      "\n⚠️  Relancer d'abord le notebook EndToEnd (cellules 0 → train_pairs → modèle v3)")

=== Vérification des prérequis ===
  ✅ model (v3): ok
  ✅ mat_skg: (53897, 2812)
  ✅ mat_semb: (53897, 2812)
  ✅ mat_p: (53897, 2812)
  ✅ test_col: (53897,)
  ✅ valid_col: (53897,)
  ✅ mask_test: (53897, 2812)
  ✅ mask_valid: (53897, 2812)
  ✅ train_pairs: (1876701, 2)
  ✅ test_u_idx: (53897,)

✅ Tout prêt — on peut construire la fusion attention


In [16]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np, math, time

class LLMKGERAttention(nn.Module):
    """
    Extension du modèle v3 avec fusion par attention adaptative.
    Les embeddings sont initialisés depuis le modèle v3 (affinés).
    La couche d attention est ajoutée par-dessus.
    """
    def __init__(self, v3_model):
        super().__init__()
        # Copier les embeddings affinés du v3 (non figés — on continue le fine-tuning)
        self.user_kg = nn.Embedding.from_pretrained(
            v3_model.user_kg.weight.data.clone(), freeze=False)
        self.item_kg = nn.Embedding.from_pretrained(
            v3_model.item_kg.weight.data.clone(), freeze=False)
        self.user_cf = nn.Embedding.from_pretrained(
            v3_model.user_cf.weight.data.clone(), freeze=False)
        self.item_cf = nn.Embedding.from_pretrained(
            v3_model.item_cf.weight.data.clone(), freeze=False)
        # Copier la projection KG (256->64) du v3
        self.kg_proj = nn.Linear(v3_model.kg_proj.in_features,
                                 v3_model.kg_proj.out_features, bias=False)
        self.kg_proj.weight.data = v3_model.kg_proj.weight.data.clone()
        # Statistiques de normalisation (fixes, du v3)
        self.register_buffer('kg_mean', v3_model.kg_mean.clone())
        self.register_buffer('kg_std',  v3_model.kg_std.clone())
        self.register_buffer('emb_mean', v3_model.emb_mean.clone())
        self.register_buffer('emb_std',  v3_model.emb_std.clone())
        # ===== NOUVEAU : couche d attention =====
        # Entrée : concat [u_kg_proj, i_kg_proj, u_cf, i_cf] = 64*4 = 256 dims
        # Sortie : 3 logits -> softmax -> (w_kg, w_emb, w_p)
        INPUT_DIM = 64 * 4
        self.attn = nn.Sequential(
            nn.Linear(INPUT_DIM, 64),
            nn.ReLU(),
            nn.Linear(64, 3),   # 3 poids : w_kg, w_emb, w_p
        )
        # Ancre pour L_KG (embeddings v3 initiaux)
        self.register_buffer('anchor_user_kg', v3_model.user_kg.weight.data.clone())
        self.register_buffer('anchor_item_kg', v3_model.item_kg.weight.data.clone())

    def _signals(self, u, i):
        """Calcule S_KG et S_emb normalisés pour un batch."""
        pu = self.kg_proj(self.user_kg(u))
        pi = self.kg_proj(self.item_kg(i))
        skg  = (pu * pi).sum(-1)
        skg  = (skg  - self.kg_mean)  / (self.kg_std  + 1e-6)
        semb = (self.user_cf(u) * self.item_cf(i)).sum(-1)
        semb = (semb - self.emb_mean) / (self.emb_std + 1e-6)
        return skg, semb, pu, pi

    def _weights(self, u, i):
        """Calcule les poids d attention adaptatifs pour chaque paire (u,i)."""
        pu = self.kg_proj(self.user_kg(u))     # [batch, 64]
        pi = self.kg_proj(self.item_kg(i))     # [batch, 64]
        ucf = self.user_cf(u)                  # [batch, 64]
        icf = self.item_cf(i)                  # [batch, 64]
        context = torch.cat([pu, pi, ucf, icf], dim=-1)  # [batch, 256]
        logits = self.attn(context)            # [batch, 3]
        w = F.softmax(logits, dim=-1)          # [batch, 3] -> somme=1
        return w[:, 0], w[:, 1], w[:, 2]      # w_kg, w_emb, w_p

    def score(self, u, i, p):
        """Score adaptatif pour chaque paire (u,i)."""
        skg, semb, _, _ = self._signals(u, i)
        w_kg, w_emb, w_p = self._weights(u, i)
        return w_kg * skg + w_emb * semb - w_p * p

    def score_and_weights(self, u, i, p):
        """Score + poids (pour analyse/debug)."""
        skg, semb, _, _ = self._signals(u, i)
        w_kg, w_emb, w_p = self._weights(u, i)
        score = w_kg * skg + w_emb * semb - w_p * p
        return score, w_kg.mean().item(), w_emb.mean().item(), w_p.mean().item()

# Créer le modèle attention depuis le v3
model_attn = LLMKGERAttention(model).to(DEVICE)
n_params = sum(p.numel() for p in model_attn.parameters() if p.requires_grad)
print(f"Modèle attention créé.")
print(f"  Paramètres entraînables : {n_params:,}")
print(f"  Couche attention : 256 → 64 → 3 (w_kg, w_emb, w_p)")
print(f"  Embeddings initialisés depuis v3 (affinés)")

# Vérifier les poids initiaux sur quelques paires
model_attn.eval()
with torch.no_grad():
    u_t = torch.tensor([1,2,3], device=DEVICE)
    i_t = torch.tensor([100,200,300], device=DEVICE)
    p_t = torch.zeros(3, device=DEVICE)
    _, wk, we, wp = model_attn.score_and_weights(u_t, i_t, p_t)
print(f"\nPoids initiaux (moyens sur 3 paires) :")
print(f"  w_kg={wk:.3f}  w_emb={we:.3f}  w_p={wp:.3f}  (somme={wk+we+wp:.3f})")

Modèle attention créé.
  Paramètres entraînables : 18,180,547
  Couche attention : 256 → 64 → 3 (w_kg, w_emb, w_p)
  Embeddings initialisés depuis v3 (affinés)

Poids initiaux (moyens sur 3 paires) :
  w_kg=0.295  w_emb=0.395  w_p=0.310  (somme=1.000)


In [17]:
GAMMA_ATT = 0.1
opt_attn = torch.optim.Adam(model_attn.parameters(), lr=0.0005)
EPOCHS_ATT = 12; BATCH = 2048
n = len(train_pairs)
history_attn = []

print(f"Entraînement attention — {EPOCHS_ATT} époques, batch={BATCH}, lr=0.0005")
print(f"{'Ep':>4} | {'L_total':>9} | {'L_rec':>9} | {'L_KG':>9} | {'w_kg':>7} {'w_emb':>7} {'w_p':>7} | {'t':>6}")
print("-"*80)

for ep in range(1, EPOCHS_ATT+1):
    model_attn.train()
    perm = np.random.permutation(n)
    tot=0.0; tot_rec=0.0; tot_kg=0.0; nb=0; t0=time.time()
    wk_ep=0.0; we_ep=0.0; wp_ep=0.0
    for s in range(0, n, BATCH):
        idx = perm[s:s+BATCH]
        u    = train_pairs[idx, 0]
        ipos = train_pairs[idx, 1]
        ineg = np.array([sample_neg(uu) for uu in u], dtype=np.int64)
        ut   = torch.tensor(u,    device=DEVICE)
        ipt  = torch.tensor(ipos, device=DEVICE)
        int_ = torch.tensor(ineg, device=DEVICE)
        p_pos = p_batch_fast(u, ipos)
        p_neg = p_batch_fast(u, ineg)

        spos = model_attn.score(ut, ipt, p_pos)
        sneg = model_attn.score(ut, int_, p_neg)
        L_rec = -torch.log(torch.sigmoid(spos - sneg) + 1e-9).mean()

        du = model_attn.user_kg(ut)  - model_attn.anchor_user_kg[ut]
        di = model_attn.item_kg(ipt) - model_attn.anchor_item_kg[ipt]
        dn = model_attn.item_kg(int_) - model_attn.anchor_item_kg[int_]
        L_KG = (du.pow(2).sum(-1).mean() + di.pow(2).sum(-1).mean() + dn.pow(2).sum(-1).mean()) / 3.0

        loss = L_rec + GAMMA_ATT * L_KG
        opt_attn.zero_grad(); loss.backward(); opt_attn.step()
        tot += loss.item(); tot_rec += L_rec.item(); tot_kg += L_KG.item(); nb += 1
        with torch.no_grad():
            _, wk, we, wp = model_attn.score_and_weights(ut, ipt, p_pos)
            wk_ep += wk; we_ep += we; wp_ep += wp

    history_attn.append(tot/nb)
    print(f"{ep:>4} | {tot/nb:>9.4f} | {tot_rec/nb:>9.4f} | {tot_kg/nb:>9.4f} | "
          f"{wk_ep/nb:>7.3f} {we_ep/nb:>7.3f} {wp_ep/nb:>7.3f} | {time.time()-t0:>5.0f}s")

print("\n✅ Entraînement attention terminé")

Entraînement attention — 12 époques, batch=2048, lr=0.0005
  Ep |   L_total |     L_rec |      L_KG |    w_kg   w_emb     w_p |      t
--------------------------------------------------------------------------------
   1 |    0.1106 |    0.1106 |    0.0001 |   0.004   0.991   0.005 |   689s
   2 |    0.1062 |    0.1062 |    0.0000 |   0.000   1.000   0.000 |   657s
   3 |    0.1039 |    0.1039 |    0.0000 |   0.000   1.000   0.000 |   538s


KeyboardInterrupt: 